In [1]:

import sys, subprocess, importlib.util, json, zipfile, hashlib, logging, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

for module, package in {
    "numpy": "numpy",
    "pandas": "pandas",
    "networkx": "networkx",
    "plotly": "plotly",
    "sklearn": "scikit-learn"
}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

SEED = 42
OUT = Path("dead_pixel_ai_output")
OUT.mkdir(exist_ok=True)

logging.basicConfig(
    filename=str(OUT / "dead_pixel_ai.log"),
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True
)
logger = logging.getLogger("DEAD_PIXEL_AI")
rng = np.random.default_rng(SEED)

print("=" * 65)
print("DEAD PIXEL AI | SILENT FAILURE INTELLIGENCE ENGINE")
print("=" * 65)

PIPELINE = nx.DiGraph([
    ("source", "transform"),
    ("transform", "warehouse"),
    ("warehouse", "analytics"),
    ("analytics", "dashboard")
])
COMPONENTS = list(nx.topological_sort(PIPELINE))

CONFIG = {
    "source": {"max_age": 35},
    "transform": {"max_age": 40},
    "warehouse": {"max_age": 45},
    "analytics": {"max_age": 50},
    "dashboard": {"max_age": 60}
}

FAULTS = [
    "normal", "stale_output", "missing_records",
    "broken_dependency", "unchanged_content",
    "legitimate_quiet_period"
]

def fingerprint(value):
    return hashlib.sha256(str(value).encode()).hexdigest()[:16]

def generate_benchmark(n_cases=600, seed=SEED):
    local_rng = np.random.default_rng(seed)
    labels = np.resize(np.array(FAULTS, dtype=object), n_cases)
    local_rng.shuffle(labels)
    rows = []

    for case_num, fault in enumerate(labels):
        case_id = f"CASE-{case_num + 1:04d}"
        run_minute = case_num * 10
        quiet = fault == "legitimate_quiet_period"
        truth = int(fault not in ("normal", "legitimate_quiet_period"))
        base_records = int(local_rng.integers(950, 1100))

        for index, component in enumerate(COMPONENTS):
            expected = int(base_records * (0.96 ** index))
            actual = max(0, int(expected * local_rng.uniform(0.94, 1.04)))
            age = int(local_rng.integers(2, 18))
            upstream_verified = True
            token = f"{case_id}-{component}-{local_rng.integers(100000, 999999)}"

            if fault == "stale_output" and component == "warehouse":
                age = 180
            elif fault == "missing_records" and component == "transform":
                actual = int(expected * 0.22)
            elif fault == "broken_dependency" and component == "analytics":
                upstream_verified = False
            elif fault == "unchanged_content" and component in (
                "warehouse", "analytics", "dashboard"
            ):
                token = f"FROZEN-{component}"
            elif quiet and component in ("source", "transform"):
                expected, actual = 0, 0

            rows.append({
                "case_id": case_id,
                "run_minute": run_minute,
                "component": component,
                "fault_type": fault,
                "ground_truth": truth,
                "output_age_minutes": age,
                "expected_records": expected,
                "actual_records": actual,
                "upstream_verified": upstream_verified,
                "content_fingerprint": fingerprint(token),
                "quiet_period": quiet,
                "execution_status": "success"
            })

    return pd.DataFrame(rows)

telemetry = generate_benchmark()

def attach_content_history(df):
    df = df.sort_values(["component", "run_minute"]).copy()
    df["unchanged_streak"] = 1

    for component, indexes in df.groupby("component").groups.items():
        previous = None
        streak = 0
        for idx in indexes:
            current = df.at[idx, "content_fingerprint"]
            streak = streak + 1 if current == previous else 1
            df.at[idx, "unchanged_streak"] = streak
            previous = current

    return df

telemetry = attach_content_history(telemetry)

def analyze_row(row):
    reasons, signals = [], []
    component = row["component"]
    max_age = CONFIG[component]["max_age"]

    if row["output_age_minutes"] > max_age:
        signals.append(0.95)
        reasons.append(
            f"STALE_OUTPUT: {row['output_age_minutes']} min; limit {max_age}"
        )

    expected, actual = row["expected_records"], row["actual_records"]
    if expected > 0 and actual < expected * 0.80 and not row["quiet_period"]:
        signals.append(0.90)
        reasons.append(f"RECORD_LOSS: expected={expected}, actual={actual}")

    if not bool(row["upstream_verified"]):
        signals.append(0.98)
        reasons.append("DEPENDENCY_FAILURE: upstream not verified")

    if int(row["unchanged_streak"]) >= 3 and not row["quiet_period"]:
        signals.append(0.82)
        reasons.append(
            f"UNCHANGED_CONTENT: repeated {int(row['unchanged_streak'])} runs"
        )

    score = 1 - np.prod([1 - signal for signal in signals]) if signals else 0.0

    return pd.Series({
        "anomaly_score": round(float(score), 4),
        "predicted_anomaly": int(score >= 0.80),
        "signal_count": len(signals),
        "failure_reasons": " | ".join(reasons) if reasons else "NONE"
    })

scored = pd.concat(
    [telemetry.reset_index(drop=True),
     telemetry.apply(analyze_row, axis=1).reset_index(drop=True)],
    axis=1
)

def calculate_impact(component):
    return sorted(nx.descendants(PIPELINE, component))

scored["downstream_impact"] = scored["component"].apply(
    lambda component: ", ".join(calculate_impact(component)) or "NONE"
)
scored["downstream_count"] = scored["component"].apply(
    lambda component: len(calculate_impact(component))
)
scored["estimated_exposure"] = np.where(
    scored["predicted_anomaly"] == 1,
    scored["downstream_count"], 0
)

def calculate_metrics(y_true, y_pred, name):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1_score": f1_score(y_true, y_pred, zero_division=0),
        "specificity": cm[0, 0] / max(1, cm[0].sum())
    }

y_true = scored["ground_truth"].astype(int)
y_pred = scored["predicted_anomaly"].astype(int)
scored["baseline_prediction"] = 0

metrics = pd.DataFrame([
    calculate_metrics(y_true, y_pred, "DEAD_PIXEL_AI"),
    calculate_metrics(y_true, scored["baseline_prediction"], "STATUS_ONLY_BASELINE")
])

print("\nBENCHMARK METRICS")
print(metrics.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

print("\nCLASSIFICATION REPORT")
print(classification_report(
    y_true, y_pred,
    labels=[0, 1],
    target_names=["Healthy / quiet", "Anomaly"],
    zero_division=0
))

alerts = scored[scored["predicted_anomaly"] == 1].copy()
alerts["severity"] = np.select(
    [
        alerts["anomaly_score"] >= 0.98,
        alerts["anomaly_score"] >= 0.90,
        alerts["anomaly_score"] >= 0.80
    ],
    ["CRITICAL", "HIGH", "MEDIUM"],
    default="LOW"
)
alerts = alerts.sort_values(
    ["anomaly_score", "downstream_count"],
    ascending=[False, False]
)

alert_columns = [
    "case_id", "component", "fault_type", "severity",
    "anomaly_score", "failure_reasons",
    "downstream_impact", "estimated_exposure"
]

print("\nTOP ALERTS")
if len(alerts):
    print(alerts[alert_columns].head(20).to_string(index=False))
else:
    print("No alerts detected.")

summary = (
    scored.groupby(["fault_type", "ground_truth"])
    .agg(
        cases=("case_id", "nunique"),
        rows=("component", "size"),
        alerts=("predicted_anomaly", "sum"),
        mean_score=("anomaly_score", "mean")
    )
    .reset_index()
)

component_summary = (
    scored.groupby("component")
    .agg(
        observations=("case_id", "count"),
        alerts=("predicted_anomaly", "sum"),
        mean_score=("anomaly_score", "mean"),
        mean_downstream=("downstream_count", "mean")
    )
    .reset_index()
)

print("\nFAULT SUMMARY")
print(summary.to_string(index=False))
print("\nCOMPONENT SUMMARY")
print(component_summary.to_string(index=False))

fig1 = px.bar(
    metrics.melt(
        id_vars="model",
        value_vars=["accuracy", "precision", "recall", "f1_score"],
        var_name="metric", value_name="score"
    ),
    x="metric", y="score", color="model", barmode="group",
    title="DEAD PIXEL AI vs Status-Only Baseline"
)
fig1.update_layout(yaxis_range=[0, 1])
fig1.show()

fig2 = px.bar(
    component_summary, x="component", y="alerts",
    text="alerts", title="Alerts by Pipeline Component"
)
fig2.show()

fig3 = px.histogram(
    scored, x="anomaly_score", color="ground_truth",
    nbins=25, barmode="overlay",
    title="Anomaly Score Distribution"
)
fig3.show()

fig4 = px.scatter(
    scored, x="expected_records", y="actual_records",
    color="fault_type", symbol="predicted_anomaly",
    hover_data=["case_id", "component", "anomaly_score"],
    title="Expected vs Actual Records"
)
fig4.show()

positions = {node: (i, 0) for i, node in enumerate(COMPONENTS)}
edge_x, edge_y = [], []
for src, dst in PIPELINE.edges():
    x0, y0 = positions[src]
    x1, y1 = positions[dst]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

fig5 = go.Figure()
fig5.add_trace(go.Scatter(
    x=edge_x, y=edge_y, mode="lines",
    line=dict(width=2), hoverinfo="none", name="Dependencies"
))
fig5.add_trace(go.Scatter(
    x=[positions[n][0] for n in COMPONENTS],
    y=[positions[n][1] for n in COMPONENTS],
    mode="markers+text", text=COMPONENTS,
    textposition="bottom center", marker=dict(size=25),
    hovertext=[
        f"{n}: {len(calculate_impact(n))} downstream component(s)"
        for n in COMPONENTS
    ],
    hoverinfo="text", name="Components"
))
fig5.update_layout(
    title="Pipeline Dependency Map", showlegend=False,
    xaxis=dict(visible=False), yaxis=dict(visible=False), height=300
)
fig5.show()

def run_tests():
    assert nx.is_directed_acyclic_graph(PIPELINE)
    assert len(scored) == 600 * len(COMPONENTS)
    assert scored["anomaly_score"].between(0, 1).all()
    assert set(scored["fault_type"].unique()) == set(FAULTS)
    assert scored["case_id"].nunique() == 600
    assert len(metrics) == 2

    test = pd.Series({
        "component": "warehouse",
        "output_age_minutes": 180,
        "expected_records": 100,
        "actual_records": 100,
        "upstream_verified": True,
        "unchanged_streak": 1,
        "quiet_period": False
    })
    assert analyze_row(test)["predicted_anomaly"] == 1

    test["output_age_minutes"] = 5
    test["actual_records"] = 10
    assert analyze_row(test)["predicted_anomaly"] == 1

    test["actual_records"] = 100
    test["upstream_verified"] = False
    assert analyze_row(test)["predicted_anomaly"] == 1

    test["upstream_verified"] = True
    test["expected_records"] = 0
    test["actual_records"] = 0
    test["quiet_period"] = True
    assert analyze_row(test)["predicted_anomaly"] == 0

    return "ALL TESTS PASSED"

test_status = run_tests()
print("\nTEST STATUS:", test_status)

scored.to_csv(OUT / "full_scored_telemetry.csv", index=False)
alerts[alert_columns].to_csv(OUT / "detected_alerts.csv", index=False)
metrics.to_csv(OUT / "benchmark_metrics.csv", index=False)
summary.to_csv(OUT / "fault_summary.csv", index=False)
component_summary.to_csv(OUT / "component_summary.csv", index=False)

report = {
    "project": "DEAD PIXEL AI",
    "version": "3.0",
    "description": "Silent Failure Intelligence Engine",
    "dataset_type": "synthetic",
    "cases": int(scored["case_id"].nunique()),
    "telemetry_rows": int(len(scored)),
    "components": COMPONENTS,
    "fault_types": FAULTS,
    "alerts": int(len(alerts)),
    "test_status": test_status,
    "metrics": metrics.to_dict(orient="records"),
    "limitations": [
        "Uses synthetic data only.",
        "Scores are heuristic, not calibrated probabilities.",
        "Repeated content can be legitimate.",
        "Benchmark results do not establish real-world performance."
    ]
}

(OUT / "executive_report.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)

(OUT / "README.txt").write_text(
    "DEAD PIXEL AI v3.0\n"
    "Silent Failure Intelligence Engine\n\n"
    "Uses synthetic telemetry to detect stale outputs, record loss, "
    "broken dependencies, and repeated content.\n"
    "Exports CSV reports, a JSON summary, logs, and charts during execution.\n"
    "This is a rule-based research prototype, not a production monitor.\n",
    encoding="utf-8"
)

zip_path = Path("DEAD_PIXEL_AI_COMPLETE.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file in OUT.rglob("*"):
        if file.is_file():
            archive.write(
                file,
                arcname=str(Path("dead_pixel_ai_output") / file.relative_to(OUT))
            )

print("\n" + "=" * 65)
print("DEAD PIXEL AI EXECUTION COMPLETE")
print("=" * 65)
print("Cases:", scored["case_id"].nunique())
print("Telemetry rows:", len(scored))
print("Alerts:", len(alerts))
print("Tests:", test_status)
print("Output directory:", OUT.resolve())
print("ZIP package:", zip_path.resolve())

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Retrieve the ZIP from your current working directory.")


DEAD PIXEL AI | SILENT FAILURE INTELLIGENCE ENGINE

BENCHMARK METRICS
               model  accuracy  precision  recall  f1_score  specificity
       DEAD_PIXEL_AI     0.435      1.000   0.153     0.265        1.000
STATUS_ONLY_BASELINE     0.333      0.000   0.000     0.000        1.000

CLASSIFICATION REPORT
                 precision    recall  f1-score   support

Healthy / quiet       0.37      1.00      0.54      1000
        Anomaly       1.00      0.15      0.27      2000

       accuracy                           0.44      3000
      macro avg       0.69      0.58      0.40      3000
   weighted avg       0.79      0.44      0.36      3000


TOP ALERTS
  case_id component        fault_type severity  anomaly_score                           failure_reasons downstream_impact  estimated_exposure
CASE-0004 analytics broken_dependency CRITICAL           0.98 DEPENDENCY_FAILURE: upstream not verified         dashboard                   1
CASE-0005 analytics broken_dependency CRITICAL 


TEST STATUS: ALL TESTS PASSED

DEAD PIXEL AI EXECUTION COMPLETE
Cases: 600
Telemetry rows: 3000
Alerts: 306
Tests: ALL TESTS PASSED
Output directory: /content/dead_pixel_ai_output
ZIP package: /content/DEAD_PIXEL_AI_COMPLETE.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>